# nnUnet results analyzer
En este notebook se analizarán los resultados obtenidos al entrenar redes de la librería nnUnet

# Importación de librerías

In [ ]:
import json
import pandas as pd
import numpy as np
import SimpleITK as sitk
import os
import matplotlib.pyplot as plt

In [ ]:
def read_dicom_series(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    return image

In [ ]:
def apply_window(image, wl, ww):
    lower = wl - ww / 2
    upper = wl + ww / 2
    windowed = np.clip(image, lower, upper)
    return windowed

# Análisis de resumen de la evaluación de los casos de validación

In [ ]:
path_2d = "C:/Users/jmriv/Downloads/summary2D.json"
path_3dfullres = "C:/Users/jmriv/Downloads/summary3D.json"

In [ ]:
with open(path_2d, 'r') as file:
    data_2d = json.load(file)

In [ ]:
with open(path_3dfullres, 'r') as file:
    data_3d = json.load(file)

In [ ]:
results = {
    "id":[],
    "DICE":[],
}
for case in data_2d['metric_per_case']:
    results['id'].append(case["reference_file"].split('\\')[-1])
    results['DICE'].append(case['metrics']['1']['Dice'])

df_2d = pd.DataFrame.from_dict(results)
df_2d.describe()

In [ ]:
results = {
    "id":[],
    "DICE":[],
}
for case in data_3d['metric_per_case']:
    results['id'].append(case["reference_file"].split('\\')[-1])
    results['DICE'].append(case['metrics']['1']['Dice'])

df_3d = pd.DataFrame.from_dict(results)
df_3d.describe()

In [ ]:
hist_2d = df_2d['DICE'].hist( bins= np.arange(0, 1, 0.05) )
hist_2d.set_title("DICE nnUNet 2D")
hist_2d.set_xlabel('Valor de índice DICE')
hist_2d.set_ylabel('Número de casos de validación')
hist_2d.set_ylim(0, 12)

In [ ]:
hist_3d = df_3d['DICE'].hist( bins= np.arange(0, 1, 0.05) )
hist_3d.set_title("DICE nnUNet 3D")
hist_3d.set_xlabel('Valor de índice DICE')
hist_3d.set_ylabel('Número de casos de validación')

In [ ]:
merged = df_2d.merge(df_3d, on='id', suffixes=('_2d', '_3d'))

In [ ]:
low_thr = merged['DICE_3d'].quantile(0.25)
high_thr = merged['DICE_3d'].quantile(0.75)

merged['group'] = pd.cut(
    merged['DICE_3d'],
    bins=[-1, low_thr, high_thr, 1],
    labels=['low', 'medium', 'high']
)

selected = (
    merged.groupby('group', group_keys=False)
          .apply(lambda x: x.sample(n=min(5, len(x)), random_state=42))
)

In [ ]:
selected

# Graficar casos

In [ ]:
ids = ['351','041','145','135', '269', '020']

In [ ]:
original_path = "D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset001_AISD/imagesTr"
labels_path = "D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset001_AISD/labelsTr"
pred_2d_path = "D:/jm.rivera/nnUnet_data/nnUNet_results/Dataset001_AISD/nnUNetTrainer__nnUNetPlans__2d/fold_0/validation"
pred_3d_path = "D:/jm.rivera/nnUnet_data/nnUNet_results/Dataset001_AISD/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/validation"

In [ ]:
wl = 40
ww = 40

for index in ids:
    ct_image = sitk.ReadImage(os.path.join(original_path, f'AISD_{index}_0000.nii.gz'))
    gt_mask = sitk.ReadImage(os.path.join(labels_path, f'AISD_{index}.nii.gz'))
    pred_mask_2d = sitk.ReadImage(os.path.join(pred_2d_path, f'AISD_{index}.nii.gz'))
    pred_mask_3d = sitk.ReadImage(os.path.join(pred_3d_path, f'AISD_{index}.nii.gz'))

    ct_np = sitk.GetArrayFromImage(ct_image)      # (z,y,x)
    mask_np = sitk.GetArrayFromImage(gt_mask)  # (z,y,x)
    mask_2d_np = sitk.GetArrayFromImage(pred_mask_2d)
    mask_3d_np = sitk.GetArrayFromImage(pred_mask_3d)

    # Find slices containing lesion
    lesion_area_per_slice = mask_np.sum(axis=(1,2))
    z = np.argmax(lesion_area_per_slice)
    windowed_slice = apply_window(ct_np[z], wl, ww)

    if lesion_area_per_slice[z] == 0:
        print("⚠️ Empty mask")
    else:
        print(f"Showing slice {z} (largest lesion area)")
        
        plt.imshow(windowed_slice, cmap="gray")
        plt.imshow(mask_np[z], alpha=0.3, cmap='Greens')
        plt.imshow(mask_2d_np[z], alpha=0.3, cmap='Reds')
        plt.imshow(mask_3d_np[z], alpha=0.3, cmap='Blues')
        plt.title(f"AISD {index} - GT (Verde) | 2D (rojo) | 3Dfullres (azul)")
        plt.show()

    fig, ax = plt.subplots(1,4, figsize=(16,4))

    ax[0].imshow(ct_np, cmap='gray')
    ax[0].set_title("Image")

    ax[1].imshow(ct_np, cmap='gray')
    ax[1].imshow(mask_np, alpha=0.4, cmap='Greens')
    ax[1].set_title("GT")

    ax[2].imshow(ct_np, cmap='gray')
    ax[2].imshow(mask_2d_np, alpha=0.4, cmap='Reds')
    ax[2].set_title("Modelo 2D")

    ax[3].imshow(ct_np, cmap='gray')
    ax[3].imshow(mask_3d_np, alpha=0.4, cmap='Blues')
    ax[3].set_title("Model 3D")

    for a in ax:
        a.axis('off')

    plt.tight_layout()
    plt.show()